# MERIT-UNETR++: GitHub and Google Colab setup

Choose **Runtime > Change runtime type > GPU**, connect, and run the cells in order. Code comes from GitHub; the 199 CT/mask pairs and manifest stay on Google Drive. The notebook does not start training.

The repository is updated only when the existing checkout is clean and on the expected branch. Record the printed commit for reproducibility and do not update it in the middle of an experiment.

In [2]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/torusama/merit-covid-ct.git'
BRANCH = 'main'
PROJECT = Path('/content/merit-covid-ct')
DRIVE_DATA = Path('/content/drive/MyDrive/PROJECT HCMUS/Paper/merit-covid-colab-2026-11-08/data/covid19_20')
CONFIG = 'configs/merit_index.yaml'

def run(*args, cwd=None):
    return subprocess.run(list(args), cwd=cwd, check=True, text=True)

## 1. Clone or safely update the GitHub checkout
Run this again after a runtime reset. An existing checkout with local changes or a different branch is left untouched. If the repository is private, authenticate Git separately; do not put tokens in this notebook.

In [3]:
if not PROJECT.exists():
    run('git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT))
else:
    if not (PROJECT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT} exists but is not a Git checkout')
    def git_output(*args):
        return subprocess.check_output(['git', *args], cwd=PROJECT, text=True).strip()
    if git_output('remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('Existing checkout has a different origin; inspect it before updating')
    if git_output('branch', '--show-current') != BRANCH:
        raise RuntimeError('Existing checkout is on a different branch; inspect it before updating')
    if git_output('status', '--porcelain'):
        raise RuntimeError('Existing checkout has local changes; preserve or commit them before updating')
    run('git', 'pull', '--ff-only', 'origin', BRANCH, cwd=PROJECT)
os.chdir(PROJECT)
run('git', 'log', '-1', '--oneline', '--decorate', cwd=PROJECT)
print('Project:', PROJECT)

Project: /content/merit-covid-ct


## 2. Connect Google Drive and check the dataset
Upload all 398 CT/mask files before continuing. Dataset completeness is checked against the saved manifest by name; this is not a checksum or a patient-identity check.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
manifest_path = DRIVE_DATA / 'cases.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Manifest not found: {manifest_path}')
records = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
if isinstance(records, dict):
    records = records['cases']
missing = []
for record in records:
    for key in ('image', 'mask' if 'mask' in record else 'infection_mask'):
        path = DRIVE_DATA / record[key]
        if not path.is_file():
            missing.append(str(path.relative_to(DRIVE_DATA)))
print(f'Manifest: {len(records)} cases; missing CT/mask files: {len(missing)}')
if len(records) != 199 or missing:
    print('Missing files:', *missing, sep='\n  ')
    raise RuntimeError('Finish uploading the 199 CT/mask pairs before continuing')
link = PROJECT / 'data/covid19_20'
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    if link.resolve() != DRIVE_DATA.resolve():
        raise RuntimeError(f'Existing dataset link points elsewhere: {link}')
elif link.exists():
    raise RuntimeError(f'Existing dataset path will not be overwritten: {link}')
else:
    link.symlink_to(DRIVE_DATA, target_is_directory=True)
print('Dataset linked:', link, '->', link.resolve())

Mounted at /content/drive
Manifest: 199 cases; missing CT/mask files: 0
Dataset linked: /content/merit-covid-ct/data/covid19_20 -> /content/drive/MyDrive/PROJECT HCMUS/Paper/merit-covid-colab-2026-11-08/data/covid19_20


## 3. Install dependencies and verify the saved split
Installation runs once per Colab runtime. This check reads split metadata and verifies paths; it does not train or tune on test data.

In [5]:
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[unetrpp]', cwd=PROJECT)
run(sys.executable, 'scripts/merit_pipeline.py', 'check-split', '--config', CONFIG, cwd=PROJECT)

CompletedProcess(args=['/usr/bin/python3', 'scripts/merit_pipeline.py', 'check-split', '--config', 'configs/merit_index.yaml'], returncode=0)

## 4. Check the GPU
The GPU is inspected in a short-lived process so the notebook kernel does not keep an extra CUDA context during profiling.

In [6]:
run(sys.executable, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')")

CompletedProcess(args=['/usr/bin/python3', '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"], returncode=0)

## Optional: profile 256x256 on this GPU
This runs a few synthetic optimizer steps, not full training. An out-of-memory result means this configuration did not fit the current runtime. Reports are temporary and contain no model checkpoint.

In [7]:
import tempfile
gpu_check = subprocess.run([sys.executable, '-c', 'import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)'])
if gpu_check.returncode != 0:
    raise RuntimeError('No GPU detected; select a GPU runtime first')
profile_path = Path(tempfile.mkdtemp(prefix='merit_gpu_')) / 'profile.json'
result = subprocess.run([sys.executable, 'scripts/merit_preflight.py', 'profile',
    '--config', CONFIG, '--size', '256', '--batch-size', '1', '--device', 'cuda',
    '--output', str(profile_path)], cwd=PROJECT)
if not profile_path.exists():
    raise RuntimeError(f'Profile failed before writing a report (exit {result.returncode})')
report = json.loads(profile_path.read_text(encoding='utf-8'))
print('Status:', report['status'])
print('Peak allocated MiB:', report.get('peak_allocated_mib'))
print('Mean step seconds:', report.get('mean_step_seconds'))
if report.get('error'):
    print('Error:', report['error'])

Status: PASS
Peak allocated MiB: 3565.9697265625
Mean step seconds: 0.9335558804999664


In [1]:
!nvidia-smi

Mon Nov  9 00:02:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

The checked-in experiment currently uses native intensity and 128x128. A fitted normalization policy and a real-CT pilot are still needed before formal training. For future training, save checkpoints to Drive, preserve the same resolved paths, and resume at epoch boundaries.